In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score
import seaborn as sns

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16 #Change to 32 for cloud platforms

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # important for pretrained CNN
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

data_dir = "D:\Study\PVR_Lab\Transformer\chest_xray"

train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=val_test_transform)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

class_names = train_dataset.classes

In [ ]:
class LiteGatedTransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

        self.gate = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid()
        )

        self.norm1 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)

        gate = self.gate(x)
        x = gate * attn_output  # YOUR CORE CONTRIBUTION

        x = self.norm1(x + attn_output)

        ff_out = self.ffn(x)
        x = self.norm2(x + ff_out)

        return x

In [ ]:
class HybridLGT(nn.Module):
    def __init__(self, num_classes=2, embed_dim=256):
        super().__init__()

        # 🔹 CNN Backbone (small change, big impact)
        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])  # remove FC

        self.pool = nn.AdaptiveAvgPool2d((14, 14))

        self.flatten_dim = 512

        self.projection = nn.Linear(self.flatten_dim, embed_dim)

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        self.transformer = LiteGatedTransformerBlock(embed_dim, num_heads=4, ff_dim=512)

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x)

        B, C, H, W = x.shape
        x = x.view(B, C, H*W).permute(0, 2, 1)  # tokens

        x = self.projection(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.transformer(x)

        cls_out = x[:, 0]
        return self.classifier(cls_out)

In [ ]:
class LGT_NoCNN(nn.Module):
    def __init__(self, num_classes=2, embed_dim=256):
        super().__init__()

        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=16, stride=16)

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        self.transformer = LiteGatedTransformerBlock(embed_dim, num_heads=4, ff_dim=512)

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)  # [B, C, H, W]

        B, C, H, W = x.shape
        x = x.flatten(2).permute(0, 2, 1)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.transformer(x)

        cls_out = x[:, 0]
        return self.classifier(cls_out)

In [ ]:
class Transformer_NoGate(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

        self.norm1 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )

        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)

        x = self.norm1(x + attn_output)

        ff_out = self.ffn(x)
        x = self.norm2(x + ff_out)

        return x

In [ ]:
class LGT_NoGate(nn.Module):
    def __init__(self, num_classes=2, embed_dim=256):
        super().__init__()

        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])

        self.pool = nn.AdaptiveAvgPool2d((14, 14))

        self.projection = nn.Linear(512, embed_dim)

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        self.transformer = Transformer_NoGate(embed_dim, num_heads=4, ff_dim=512)

        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x)

        B, C, H, W = x.shape
        x = x.view(B, C, H*W).permute(0, 2, 1)

        x = self.projection(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.transformer(x)

        cls_out = x[:, 0]
        return self.classifier(cls_out)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HybridLGT().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_model(model, train_loader, val_loader, epochs=10):
    model = model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # validation accuracy
        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                correct += (outputs.argmax(1) == labels).sum().item()

        val_acc = correct / len(val_dataset)

        if val_acc > best_acc:
            best_acc = val_acc

        print(f"Epoch {epoch+1} - Val Acc: {val_acc:.4f}")

    return best_acc

In [ ]:
print("\n================ ABLATION STUDY ================\n")

results = {}

print("Running Full Model (LGT + CNN)...")
results["LGT + CNN"] = train_model(HybridLGT(), train_loader, val_loader)

print("\nRunning LGT without CNN...")
results["LGT Only"] = train_model(LGT_NoCNN(), train_loader, val_loader)

print("\nRunning LGT without Gate...")
results["No Gate"] = train_model(LGT_NoGate(), train_loader, val_loader)

print("\nFinal Results:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")